# Titanic Survival Prediction — Version 3
**Goal:** Improve generalization with feature engineering, cross-validation, and model comparison. **85% Kaggle accuracy is an aspiration, not a guaranteed outcome.**

## 1. Load data
Place `train.csv` and `test.csv` beside this notebook, or adjust the paths. The hidden Kaggle labels are never used for training or model selection.

In [2]:
%pip install catboost
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from catboost import CatBoostClassifier
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
print("Train:", train.shape, "Test:", test.shape)
display(train.head())


   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   - -------------------------------------- 4.2/100.2 MB 22.1 MB/s eta 0:00:05
   --- ------------------------------------ 9.2/100.2 MB 22.9 MB/s eta 0:00:04
   ----- ---------------------------------- 13.1/100.2 MB 21.7 MB/s eta 0:00:05
   ------ --------------------------------- 17.0/100.2 MB 22.6 MB/s eta 0:00:04
   ------- -------------------------------- 19.9/100.2 MB 19.8 MB/s eta 0:00:05
   --------- ------------------------------ 23.3/100.2 MB 19.1 MB/s eta 0:00:05
   ---------- ----------------------------- 26.5/100.2 MB 18.3 MB/s eta 0:00:05
   ------------ --------------------------- 30.4/100.2 MB 18.5 MB/s eta 0:00:04
   ------------- -------------------------- 34.3/100.2 MB 18.5 MB/s eta 0:00:04
   --------------- ------------------------ 38.8/100.2 MB 18.6 MB/s eta 0:00:04
   ---------------- ----------------------- 42.5/100.2 MB 18.5 MB/s eta 0:00:04
   ------------------ --------------------- 47.4/1


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Train: (891, 12) Test: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. Exploratory data analysis

In [3]:
display(train.describe(include="all").T)
display(train.isna().sum().sort_values(ascending=False).to_frame("Missing"))
print("Survival rate:", round(train.Survived.mean(), 3))

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
PassengerId,891.0,NaN,NaN,NaN,446.0,257.353842,1.0,223.5,446.0,668.5,891.0
Survived,891.0,NaN,NaN,NaN,0.383838,0.486592,0.0,0.0,0.0,1.0,1.0
Pclass,891.0,NaN,NaN,NaN,2.308642,0.836071,1.0,2.0,3.0,3.0,3.0
Name,891,891,"Dooley, Mr. Patrick",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sex,891,2,male,577,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,714.0,NaN,NaN,NaN,29.699118,14.526497,0.42,20.125,28.0,38.0,80.0
SibSp,891.0,NaN,NaN,NaN,0.523008,1.102743,0.0,0.0,0.0,1.0,8.0
Parch,891.0,NaN,NaN,NaN,0.381594,0.806057,0.0,0.0,0.0,0.0,6.0
Ticket,891,681,347082,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Fare,891.0,NaN,NaN,NaN,32.204208,49.693429,0.0,7.9104,14.4542,31.0,512.3292


,Missing
Cabin,687
Age,177
Embarked,2
PassengerId,0
Name,0
Pclass,0
Survived,0
Sex,0
Parch,0
SibSp,0


Survival rate: 0.384


In [11]:
%pip install --upgrade --force-reinstall matplotlib
import matplotlib
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot([1, 2, 3, 4], [10, 20, 15, 30])
plt.title("Matplotlib Test")
plt.xlabel("X Axis")
plt.ylabel("Y Axis")
plt.show()


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
error: uninstall-no-record-file

Cannot uninstall matplotlib 3.11.1

The package's contents are unknown: no RECORD file was found for matplotlib.

hint: You might be able to recover from this via: pip install --force-reinstall --no-deps matplotlib==3.11.1



  Using cached packaging-26.3-py3-none-any.whl.metadata (3.5 kB)
  Using cached pillow-12.3.0-cp313-cp313-win_amd64.whl.metadata (9.3 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   -------- ------------------------------- 2.1/9.3 MB 11.6 MB/s eta 0:00:01
   -------------------- ------------------- 4.7/9.3 MB 11.6 MB/s eta 0:00:01
   ------------------------------ --------- 7.1/9.3 MB 11.7 MB/s eta 0:00:01
   ---------------------------------------  9.2/9.3 MB 11.8 MB/s eta 0:00:01
   ---------------------------------------- 9.3/9.3 MB 10.3 MB/s  0:00:00
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ----------------------------- ---------- 1.8/2.5 MB 10.2 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 7.0 MB/s  0:00:00
   ---------------------------------------- 

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\adhil\\anaconda3\\Lib\\site-packages\\matplotlib\\mpl-data\\fonts\\ttf\\LastResortHE-Regular.ttf'

<Figure size 800x500 with 1 Axes>

## 3. Feature engineering
Create family size, traveling alone, titles, ticket prefix, cabin deck, and fare per person. Missing numeric values are imputed **within each CV training fold** to prevent leakage.

In [12]:
def features(df):
 x=pd.DataFrame(index=df.index)
 x['Pclass']=df.Pclass;x['Sex']=df.Sex.map({'male':0,'female':1});x['Age']=df.Age;x['Fare']=df.Fare
 x['SibSp']=df.SibSp;x['Parch']=df.Parch;x['FamilySize']=df.SibSp+df.Parch+1;x['IsAlone']=(x.FamilySize==1).astype(int)
 x['Embarked']=df.Embarked.map({'S':0,'C':1,'Q':2});x['Title']=df.Name.str.extract(r',\s*([^.]*)\.')[0].str.strip().replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})
 x['Title']=x.Title.where(x.Title.isin(['Mr','Mrs','Miss','Master']), 'Rare').map({'Mr':0,'Mrs':1,'Miss':2,'Master':3,'Rare':4})
 x['Deck']=df.Cabin.fillna('U').str[0].map({v:i for i,v in enumerate('ABCDEFGTU')})
 x['TicketPrefix']=df.Ticket.str.replace(r'[./\d\s]','',regex=True).str.upper().replace('', 'NONE')
 common=['NONE','PC','CA','A','STON','SOTON','SCPARIS','WC'];x['TicketPrefix']=x.TicketPrefix.where(x.TicketPrefix.isin(common),'OTHER').map({v:i for i,v in enumerate(common+['OTHER'])})
 x['FarePerPerson']=x.Fare/x.FamilySize
 x['Child']=(x.Age<12).astype(int)
 return x
X=features(train); X_test=features(test); y=train["Survived"]
display(X.head())

,Pclass,Sex,Age,Fare,SibSp,Parch,FamilySize,IsAlone,Embarked,Title,Deck,TicketPrefix,FarePerPerson,Child
0,3,0,22.0,7.2500,1,0,2,0,0.0,0,8,3,3.62500,0
1,1,1,38.0,71.2833,1,0,2,0,1.0,1,2,1,35.64165,0
2,3,1,26.0,7.9250,0,0,1,1,0.0,2,8,8,7.92500,0
3,1,1,35.0,53.1000,1,0,2,0,0.0,1,2,0,26.55000,0
4,3,0,35.0,8.0500,0,0,1,1,0.0,0,8,0,8.05000,0


## 4. Compare three models with five-fold stratified CV

In [13]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
models={
 'ExtraTrees':make_pipeline(SimpleImputer(strategy='median'),ExtraTreesClassifier(n_estimators=450,max_features=.8,min_samples_leaf=2,random_state=42,n_jobs=2)),
 'RandomForest':make_pipeline(SimpleImputer(strategy='median'),RandomForestClassifier(n_estimators=450,max_depth=7,min_samples_leaf=2,max_features='sqrt',random_state=42,n_jobs=2)),
 'HistGradientBoosting':make_pipeline(SimpleImputer(strategy='median'),HistGradientBoostingClassifier(max_iter=180,max_leaf_nodes=12,learning_rate=.045,l2_regularization=3,random_state=42))}

In [14]:
results=[]
for name, estimator in models.items():
    scores=cross_validate(estimator,X,y,cv=cv,scoring=["accuracy","f1"],n_jobs=1)
    results.append((name,scores["test_accuracy"].mean(),scores["test_accuracy"].std(),scores["test_f1"].mean()))
comparison=pd.DataFrame(results,columns=["Model","CV Accuracy","CV Std","CV F1"]).sort_values("CV Accuracy",ascending=False)
display(comparison)
comparison.to_csv("Titanic_v3_model_comparison.csv",index=False)

C:\Users\adhil\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\adhil\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Users\adhil\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\adhil\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^

,Model,CV Accuracy,CV Std,CV F1
2,HistGradientBoosting,0.837223,0.021548,0.777898
1,RandomForest,0.836125,0.009225,0.774786
0,ExtraTrees,0.827142,0.019734,0.764474


In [15]:
comparison.plot.bar(x="Model",y="CV Accuracy",legend=False,figsize=(9,4),ylim=(0.65,0.9),rot=20)
plt.ylabel("5-fold CV accuracy");plt.tight_layout();plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\adhil\\anaconda3\\Lib\\site-packages\\matplotlib\\mpl-data\\fonts\\ttf\\LastResortHE-Regular.ttf'

Error in callback <function _draw_all_if_interactive at 0x000001A495D5D260> (for post_execute), with arguments args (),kwargs {}:


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\adhil\\anaconda3\\Lib\\site-packages\\matplotlib\\mpl-data\\fonts\\ttf\\LastResortHE-Regular.ttf'

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\adhil\\anaconda3\\Lib\\site-packages\\matplotlib\\mpl-data\\fonts\\ttf\\LastResortHE-Regular.ttf'

<Figure size 900x400 with 1 Axes>

## 5. Out-of-fold error analysis
Predictions here are generated from models that have not trained on the corresponding fold.

In [17]:
best_name=comparison.iloc[0]["Model"]
best_model=models[best_name]
oof=cross_val_predict(best_model,X,y,cv=cv,n_jobs=1)
print("Selected model:",best_name)
print(classification_report(y,oof))
ConfusionMatrixDisplay.from_predictions(y,oof);plt.show()

Selected model: HistGradientBoosting
              precision    recall  f1-score   support

           0       0.85      0.90      0.87       549
           1       0.82      0.74      0.78       342

    accuracy                           0.84       891
   macro avg       0.83      0.82      0.82       891
weighted avg       0.84      0.84      0.84       891



FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\adhil\\anaconda3\\Lib\\site-packages\\matplotlib\\mpl-data\\fonts\\ttf\\LastResortHE-Regular.ttf'

<Figure size 640x480 with 2 Axes>

## 6. Fit selected model and export Kaggle predictions
The Kaggle test set has no labels. Its score is only available after submitting the CSV.

In [18]:
best_model.fit(X,y)
test_predictions=best_model.predict(X_test).astype(int)
submission=pd.DataFrame({"PassengerId":test["PassengerId"],"Survived":test_predictions})
assert submission.shape==(418,2)
assert set(submission.Survived.unique()).issubset({0,1})
submission.to_csv("submission_v3.csv",index=False)
display(submission.head())
print("Saved submission_v3.csv")

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


Saved submission_v3.csv


## 7. Interpretation and next steps
Compare your new Kaggle score to 0.77751.